In [ ]:
import os, glob, time, json, gc
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from transformers import WhisperModel, WhisperFeatureExtractor

MODEL_ID = 'openai/whisper-large-v3'
BASE = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
WORK = '/kaggle/working'

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
uids = ['train/' + f for f in train.filename] + ['test/' + f for f in test.filename]
assert len(set(uids)) == len(uids) == 985, 'unexpected number of files'
print(len(uids), 'files | cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU is off: set Accelerator to GPU T4 x2 in session options'

device = 'cuda'
fe = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
_full = WhisperModel.from_pretrained(MODEL_ID)
enc = _full.encoder.half().to(device).eval()
del _full
gc.collect()
print('encoder loaded | layers:', enc.config.encoder_layers, '| dim:', enc.config.d_model,
      '| mel bins:', fe.feature_size)

In [ ]:
# E2: embedding function aur smoke test
SR = 16000
CH = 30 * SR      # encoder window: 30 s
FRAME = 320       # one encoder frame covers 320 samples (50 frames per second)

@torch.inference_mode()
def embed(uid):
    y, sr = sf.read(f'{BASE}/{uid}', dtype='float32')
    if y.ndim > 1:
        y = y.mean(axis=1)
    assert sr == SR, f'unexpected sample rate {sr} for {uid}'
    chunks = [y[i:i + CH] for i in range(0, len(y), CH)]
    if len(chunks) > 1 and len(chunks[-1]) < 2 * SR:     # drop a tiny tail
        chunks = chunks[:-1]
    feats = fe(chunks, sampling_rate=SR, return_tensors='pt').input_features.to(device).half()
    hs = enc(feats, output_hidden_states=True).hidden_states     # tuple of (chunks, 1500, dim)
    H = torch.stack(hs, 0)                                       # (layers, chunks, 1500, dim), fp16
    s1 = s2 = None
    n_tot = 0
    for b, c in enumerate(chunks):
        nv = max(1, min(1500, (len(c) + FRAME - 1) // FRAME))    # frames covering real audio only
        Hb = H[:, b, :nv].float()
        s1 = Hb.sum(1) if s1 is None else s1 + Hb.sum(1)
        s2 = (Hb ** 2).sum(1) if s2 is None else s2 + (Hb ** 2).sum(1)
        n_tot += nv
    mean = s1 / n_tot
    std = (s2 / n_tot - mean ** 2).clamp_min(0).sqrt()
    return mean.cpu().numpy(), std.cpu().numpy(), n_tot

# three paths: a 45 s file (2 chunks), a 60 s file (tail dropped), a 6.5 s file (1 short chunk)
for u in ['train/audio_126.wav', 'train/audio_192.wav', 'test/audio_107.wav']:
    m, s, n = embed(u)
    print(u, '| frames pooled:', n, '| shapes:', m.shape, s.shape)
print('expected frames pooled: about 2253, 3000, 324')

a_m, a_s, _ = embed('train/audio_126.wav')
b_m, b_s, _ = embed('train/audio_126.wav')
print('repeat-run max abs diff (mean, std):', float(np.abs(a_m - b_m).max()), float(np.abs(a_s - b_s).max()))
print('NaN or inf present:', not (np.isfinite(a_m).all() and np.isfinite(a_s).all()))
print('largest abs feature value:', round(float(max(np.abs(a_m).max(), np.abs(a_s).max())), 1),
      '(must stay well below 65504, the fp16 limit)')

t0 = time.time()
for u in uids[:5]:
    embed(u)
per_file = (time.time() - t0) / 5
print('seconds per file:', round(per_file, 2), '| estimated total minutes:', round(per_file * 985 / 60, 1))
print('peak GPU memory GB:', round(torch.cuda.max_memory_allocated() / 1e9, 2))

In [ ]:
N, L, D = len(uids), enc.config.encoder_layers + 1, enc.config.d_model
P_MEAN, P_STD, P_FR, P_N = (f'{WORK}/_part_mean.npy', f'{WORK}/_part_std.npy',
                            f'{WORK}/_part_frames.npy', f'{WORK}/_part_n.json')

if os.path.exists(P_N):
    feats_mean, feats_std, frames = np.load(P_MEAN), np.load(P_STD), np.load(P_FR)
    start = json.load(open(P_N))['n_done']
    print('resuming from', start)
else:
    feats_mean = np.zeros((N, L, D), dtype=np.float16)
    feats_std = np.zeros((N, L, D), dtype=np.float16)
    frames = np.zeros(N, dtype=np.int32)
    start = 0

t0 = time.time()
for i in range(start, N):
    m, s, n = embed(uids[i])
    feats_mean[i], feats_std[i], frames[i] = m, s, n
    if (i + 1) % 100 == 0 or i + 1 == N:
        np.save(P_MEAN, feats_mean); np.save(P_STD, feats_std); np.save(P_FR, frames)
        json.dump({'n_done': i + 1}, open(P_N, 'w'))
        el = time.time() - t0
        print(f'{i + 1}/{N} | {el / 60:.1f} min | est remaining {(N - i - 1) * el / (i + 1 - start) / 60:.1f} min')

bad = not (np.isfinite(feats_mean).all() and np.isfinite(feats_std).all())
print('NaN or inf present:', bad)
print('all-zero rows:', int((np.abs(feats_mean).sum(axis=(1, 2)) == 0).sum()))
uniq = len(np.unique(feats_mean[:, 21].astype(np.float32).round(3), axis=0))
print('unique rows at layer 21:', uniq, 'of', N)
print('frames pooled: min', int(frames.min()), '| max', int(frames.max()))

np.save(f'{WORK}/whisper_enc_mean.npy', feats_mean)
np.save(f'{WORK}/whisper_enc_std.npy', feats_std)
np.save(f'{WORK}/whisper_enc_nframes.npy', frames)
pd.Series(uids).to_csv(f'{WORK}/whisper_enc_uids.csv', index=False, header=['uid'])
for p in (P_MEAN, P_STD, P_FR, P_N):
    os.remove(p)
print('saved', feats_mean.shape, '|', sorted(os.listdir(WORK)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

norms, cos = [], []
for l in range(L):
    X = feats_mean[:, l].astype(np.float32)
    norms.append(float(np.linalg.norm(X, axis=1).mean()))
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    S = Xn @ Xn.T
    cos.append(float((S.sum() - np.trace(S)) / (N * (N - 1))))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(range(L), norms, marker='o'); ax[0].set_title('Mean L2 norm of pooled embedding'); ax[0].set_xlabel('layer')
ax[1].plot(range(L), cos, marker='o', color='tab:green'); ax[1].set_title('Mean cosine similarity between files'); ax[1].set_xlabel('layer')
plt.tight_layout(); plt.show()